# 🧄 Apache Iceberg com Apache Spark

## Cenário: Superstore — Varejo nos Estados Unidos

**Fonte de dados:** [Superstore Dataset — Kaggle (vivek468)](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)  
**Licença:** Other  
**Amostra utilizada:** 20 clientes e 20 pedidos de uma rede de varejo norte-americana

---

### Modelo ER

```
┌─────────────────────┐       ┌────────────────────────────────────┐
│      clientes       │       │             pedidos                │
│─────────────────────│       │────────────────────────────────────│
│ customer_id (PK)    │──────<│ order_id                           │
│ customer_name       │       │ customer_id (FK)                   │
│ segment             │       │ order_date                         │
│ city                │       │ ship_date                          │
│ state               │       │ ship_mode                          │
│ region              │       │ product_name                       │
└─────────────────────┘       │ category                           │
                              │ sub_category                       │
                              │ sales      DOUBLE                  │
                              │ quantity   INT                     │
                              │ discount   DOUBLE                  │
                              │ profit     DOUBLE                  │
                              └────────────────────────────────────┘
```

### DDL

```sql
CREATE TABLE clientes (
    customer_id   STRING,
    customer_name STRING,
    segment       STRING,
    city          STRING,
    state         STRING,
    region        STRING
) USING iceberg;

CREATE TABLE pedidos (
    order_id      STRING,
    customer_id   STRING,
    order_date    STRING,
    ship_date     STRING,
    ship_mode     STRING,
    product_name  STRING,
    category      STRING,
    sub_category  STRING,
    sales         DOUBLE,
    quantity      INT,
    discount      DOUBLE,
    profit        DOUBLE
) USING iceberg
PARTITIONED BY (category);
```

### Diferenciais do Iceberg
- **Particionamento oculto** — a query não precisa filtrar pela coluna de partição
- **Schema Evolution** — adicionar/remover colunas sem reescrever dados
- **Time Travel** — snapshots imutáveis por operação
- **Multi-engine** — Spark, Trino, Flink, Hive leem o mesmo formato

## 1. Configuração do Ambiente com Iceberg

In [1]:
import os
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit

CWD = os.getcwd()
PROJECT_ROOT   = os.path.dirname(CWD) if os.path.basename(CWD) == 'notebooks' else CWD
DATA_RAW       = os.path.join(PROJECT_ROOT, 'data', 'raw')
ICEBERG_PATH   = os.path.join(PROJECT_ROOT, 'data', 'iceberg')

ICEBERG_VERSION = '1.5.0'
SCALA_VERSION   = '2.12'

spark = (
    SparkSession.builder
    .appName('Iceberg - Superstore')
    .config(
        'spark.jars.packages',
        f'org.apache.iceberg:iceberg-spark-runtime-3.5_{SCALA_VERSION}:{ICEBERG_VERSION}'
    )
    .config('spark.sql.extensions',
            'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .config('spark.sql.catalog.local', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.local.type', 'hadoop')
    .config('spark.sql.catalog.local.warehouse', ICEBERG_PATH)
    .config('spark.sql.defaultCatalog', 'local')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')
print(f'PySpark: {pyspark.__version__}')
print(f'Dados CSV    : {DATA_RAW}')
print(f'Iceberg path : {ICEBERG_PATH}')
print('\u2705 SparkSession com Apache Iceberg iniciada!')

:: loading settings :: url = jar:file:/home/gabrielmaciel/apache_spark_delta_lake_apache_iceberg/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/gabrielmaciel/.ivy2/cache
The jars for the packages stored in: /home/gabrielmaciel/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a9dda83c-0817-4a20-93b9-342cdb210ebc;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 in central
:: resolution report :: resolve 450ms :: artifacts dl 26ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :

PySpark: 3.5.1
Dados CSV    : /home/gabrielmaciel/apache_spark_delta_lake_apache_iceberg/data/raw
Iceberg path : /home/gabrielmaciel/apache_spark_delta_lake_apache_iceberg/data/iceberg
✅ SparkSession com Apache Iceberg iniciada!


## 2. Leitura da Fonte de Dados — Kaggle Superstore

Os dados vêm do **Superstore Dataset** (Kaggle) — uma rede de varejo norte-americana com vendas de Furniture, Office Supplies e Technology.  
Usamos uma amostra de **20 clientes** e **20 pedidos** com dados completamente legíveis.

In [2]:
df_clientes = spark.read.csv(
    os.path.join(DATA_RAW, 'sample_clientes.csv'),
    header=True, inferSchema=True
)

df_pedidos = spark.read.csv(
    os.path.join(DATA_RAW, 'sample_pedidos.csv'),
    header=True, inferSchema=True
)

print('Clientes (amostra Superstore):')
df_clientes.show(5, truncate=False)

print('Pedidos — distribuicao por categoria:')
df_pedidos.groupBy('category').count().show()

print('Pedidos (amostra):')
df_pedidos.select('order_id', 'customer_id', 'product_name', 'category', 'sales', 'profit', 'ship_mode').show(5, truncate=True)

Clientes (amostra Superstore):
+-----------+---------------+---------+---------------+--------------+------+
|customer_id|customer_name  |segment  |city           |state         |region|
+-----------+---------------+---------+---------------+--------------+------+
|CG-12520   |Claire Gute    |Consumer |Henderson      |Kentucky      |South |
|DV-13045   |Darrin Van Huff|Corporate|Los Angeles    |California    |West  |
|SO-20335   |Sean O'Donnell |Consumer |Fort Lauderdale|Florida       |South |
|BH-11710   |Brosina Hoffman|Consumer |Los Angeles    |California    |West  |
|AA-10480   |Andrew Allen   |Consumer |Concord        |North Carolina|South |
+-----------+---------------+---------+---------------+--------------+------+
only showing top 5 rows

Pedidos — distribuicao por categoria:
+---------------+-----+
|       category|count|
+---------------+-----+
|Office Supplies|   10|
|      Furniture|    7|
|     Technology|    3|
+---------------+-----+

Pedidos (amostra):
+--------------+

## 3. Criação do Namespace e Tabelas Iceberg

In [3]:
spark.sql('CREATE NAMESPACE IF NOT EXISTS local.superstore')
print('\u2705 Namespace criado: local.superstore')

spark.sql('DROP TABLE IF EXISTS local.superstore.clientes')
spark.sql('DROP TABLE IF EXISTS local.superstore.pedidos')

spark.sql('''
    CREATE TABLE local.superstore.clientes (
        customer_id   STRING,
        customer_name STRING,
        segment       STRING,
        city          STRING,
        state         STRING,
        region        STRING
    ) USING iceberg
''')

spark.sql('''
    CREATE TABLE local.superstore.pedidos (
        order_id      STRING,
        customer_id   STRING,
        order_date    STRING,
        ship_date     STRING,
        ship_mode     STRING,
        product_name  STRING,
        category      STRING,
        sub_category  STRING,
        sales         DOUBLE,
        quantity      INT,
        discount      DOUBLE,
        profit        DOUBLE
    ) USING iceberg
    PARTITIONED BY (category)
''')

print('\u2705 Tabelas Iceberg criadas com particionamento por category!')

✅ Namespace criado: local.superstore
✅ Tabelas Iceberg criadas com particionamento por category!


## 4. INSERT — Carregando dados do Kaggle Superstore para o Iceberg

O Spark lê os CSVs e insere nas tabelas Iceberg via SQL puro.  
O Iceberg automaticamente organiza os dados nas partições por `category` (Furniture, Office Supplies, Technology).

In [4]:
df_clientes.createOrReplaceTempView('raw_clientes')
df_pedidos.createOrReplaceTempView('raw_pedidos')

spark.sql('INSERT INTO local.superstore.clientes SELECT * FROM raw_clientes')
spark.sql('INSERT INTO local.superstore.pedidos SELECT * FROM raw_pedidos')

print('\u2705 INSERT executado com dados reais do Superstore')
print('Clientes inseridos:', spark.sql('SELECT count(*) FROM local.superstore.clientes').first()[0])
print('Pedidos inseridos :', spark.sql('SELECT count(*) FROM local.superstore.pedidos').first()[0])

print('\nDistribuição por categoria (partição Iceberg):')
spark.sql('SELECT category, count(*) as total FROM local.superstore.pedidos GROUP BY category').show()

print('Distribuição por ship_mode:')
spark.sql('SELECT ship_mode, count(*) as total FROM local.superstore.pedidos GROUP BY ship_mode').show()

print('Amostra de pedidos:')
spark.sql('SELECT order_id, customer_id, product_name, category, sales, profit, ship_mode FROM local.superstore.pedidos LIMIT 5').show(truncate=True)

✅ INSERT executado com dados reais do Superstore
Clientes inseridos: 20
Pedidos inseridos : 20

Distribuição por categoria (partição Iceberg):
+---------------+-----+
|       category|total|
+---------------+-----+
|Office Supplies|   10|
|      Furniture|    7|
|     Technology|    3|
+---------------+-----+

Distribuição por ship_mode:
+--------------+-----+
|     ship_mode|total|
+--------------+-----+
|  Second Class|   10|
|Standard Class|   10|
+--------------+-----+

Amostra de pedidos:
+--------------+-----------+--------------------+---------+--------+--------+--------------+
|      order_id|customer_id|        product_name| category|   sales|  profit|     ship_mode|
+--------------+-----------+--------------------+---------+--------+--------+--------------+
|CA-2016-152156|   CG-12520|Bush Somerset Col...|Furniture|  261.96| 41.9136|  Second Class|
|CA-2016-152156|   CG-12520|Hon Deluxe Fabric...|Furniture|  731.94| 219.582|  Second Class|
|US-2017-156909|   SF-20065|Global D

## 5. UPDATE — Atualizando modo de envio

Pedidos com `ship_mode = 'Second Class'` serão atualizados para `'First Class'`.  
O Iceberg cria um novo snapshot e registra o DELETE file apontando as linhas alteradas.

In [5]:
spark.sql('''
    UPDATE local.superstore.pedidos
    SET ship_mode = 'First Class'
    WHERE ship_mode = 'Second Class'
''')

print('\u2705 UPDATE — Second Class \u2192 First Class')
spark.sql('SELECT ship_mode, count(*) as total FROM local.superstore.pedidos GROUP BY ship_mode').show()

✅ UPDATE — Second Class → First Class
+--------------+-----+
|     ship_mode|total|
+--------------+-----+
|   First Class|   10|
|Standard Class|   10|
+--------------+-----+



## 6. DELETE — Removendo vendas com prejuízo

Removemos os registros onde `profit < 0`.  
O snapshot anterior continua acessível via **Time Travel**.

In [6]:
spark.sql('''
    DELETE FROM local.superstore.pedidos
    WHERE profit < 0
''')

print('\u2705 DELETE — registros com profit < 0 removidos')
print('Registros restantes:', spark.sql('SELECT count(*) FROM local.superstore.pedidos').first()[0])
spark.sql('SELECT order_id, product_name, category, sales, profit, ship_mode FROM local.superstore.pedidos').show(truncate=True)

✅ DELETE — registros com profit < 0 removidos
Registros restantes: 14
+--------------+--------------------+---------------+--------+-------+--------------+
|      order_id|        product_name|       category|   sales| profit|     ship_mode|
+--------------+--------------------+---------------+--------+-------+--------------+
|CA-2016-152156|Bush Somerset Col...|      Furniture|  261.96|41.9136|   First Class|
|CA-2016-152156|Hon Deluxe Fabric...|      Furniture|  731.94|219.582|   First Class|
|CA-2014-115812|Eldon Expressions...|      Furniture|   48.86|14.1694|Standard Class|
|CA-2014-115812|Chromcraft Rectan...|      Furniture|1706.184|85.3092|Standard Class|
|CA-2016-138688|Self-Adhesive Add...|Office Supplies|   14.62| 6.8714|   First Class|
|CA-2014-167164|Fellowes Super St...|Office Supplies|    55.5|   9.99|   First Class|
|CA-2014-143336|          Newell 341|Office Supplies|    8.56| 2.4824|   First Class|
|US-2015-108966|Eldon Fold 'N Rol...|Office Supplies|  22.368| 2.5164|

## 7. MERGE (UPSERT) — Atualizar ou inserir em uma operação

O `MERGE` verifica se o registro já existe:  
- Se **existe** (`order_id` encontrado) → atualiza  
- Se **não existe** → insere

In [7]:
existente = spark.sql('SELECT * FROM local.superstore.pedidos LIMIT 1').first()

schema_cols = ['order_id', 'customer_id', 'order_date', 'ship_date', 'ship_mode',
               'product_name', 'category', 'sub_category',
               'sales', 'quantity', 'discount', 'profit']

dados_merge = [
    # UPDATE — pedido existente ganha quantidade +1
    (existente['order_id'], existente['customer_id'],
     existente['order_date'], existente['ship_date'], existente['ship_mode'],
     existente['product_name'], existente['category'], existente['sub_category'],
     existente['sales'], existente['quantity'] + 1, existente['discount'], existente['profit']),
    # INSERT — novo pedido
    ('US-2024-NOVO01', 'CG-12520', '1/15/2024', '1/18/2024', 'First Class',
     'Apple MacBook Pro 16"', 'Technology', 'Computers', 2499.99, 1, 0.0, 499.99),
]

df_merge = spark.createDataFrame(dados_merge, schema_cols)
df_merge.createOrReplaceTempView('novos_pedidos')

spark.sql('''
    MERGE INTO local.superstore.pedidos AS destino
    USING novos_pedidos AS origem
    ON destino.order_id = origem.order_id AND destino.product_name = origem.product_name
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
''')

print('\u2705 MERGE executado!')
spark.sql('''
    SELECT order_id, product_name, category, sales, quantity, profit, ship_mode
    FROM local.superstore.pedidos
    ORDER BY order_id
''').show(truncate=True)

✅ MERGE executado!
+--------------+--------------------+---------------+--------+--------+-------+--------------+
|      order_id|        product_name|       category|   sales|quantity| profit|     ship_mode|
+--------------+--------------------+---------------+--------+--------+-------+--------------+
|CA-2014-115812|Chromcraft Rectan...|      Furniture|1706.184|       9|85.3092|Standard Class|
|CA-2014-115812|Eldon Expressions...|      Furniture|   48.86|       7|14.1694|Standard Class|
|CA-2014-115812|          Newell 322|Office Supplies|    7.28|       4| 1.9656|Standard Class|
|CA-2014-115812|DXL Angle-View Bi...|Office Supplies|  18.504|       3| 5.7825|Standard Class|
|CA-2014-115812|Belkin F5C206VTEL...|Office Supplies|   114.9|       5|  34.47|Standard Class|
|CA-2014-115812|Mitel 5320 IP Pho...|     Technology| 907.152|       6|90.7152|Standard Class|
|CA-2014-115812|Konftel 250 Confe...|     Technology| 911.424|       4|68.3568|Standard Class|
|CA-2014-143336|          Newel

## 8. Time Travel — Snapshots do Iceberg

O Iceberg usa o conceito de **snapshots** — cada operação de escrita cria um novo snapshot imutável.  
É possível consultar o estado dos dados em qualquer snapshot anterior.

In [8]:
snapshots_df = spark.sql(
    'SELECT snapshot_id, committed_at, operation FROM local.superstore.pedidos.snapshots'
)
snapshots_df.show(truncate=False)

primeiro_snapshot = snapshots_df.orderBy('committed_at').first()['snapshot_id']

print(f'\nEstado inicial (snapshot_id={primeiro_snapshot}):')
spark.sql(f'''
    SELECT order_id, product_name, profit, ship_mode
    FROM local.superstore.pedidos VERSION AS OF {primeiro_snapshot}
''').show(truncate=True)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|4805976894671930875|2026-04-29 17:51:13.597|append   |
|2912268928200267627|2026-04-29 17:51:20.961|overwrite|
|5897705967078635686|2026-04-29 17:51:22.949|overwrite|
|4425219915812739318|2026-04-29 17:51:32.332|overwrite|
+-------------------+-----------------------+---------+


Estado inicial (snapshot_id=4805976894671930875):
+--------------+--------------------+--------+--------------+
|      order_id|        product_name|  profit|     ship_mode|
+--------------+--------------------+--------+--------------+
|CA-2016-152156|Bush Somerset Col...| 41.9136|  Second Class|
|CA-2016-152156|Hon Deluxe Fabric...| 219.582|  Second Class|
|US-2017-156909|Global Deluxe Sta...| -1.0196|  Second Class|
|CA-2016-128916|Staple-based wall...| -3.8208|  Second Class|
|US-2015-108966|Bretford CR4500 S...|-383.031|Standard Class|
|CA-

## 9. Schema Evolution — Adicionar coluna sem reescrever dados

Um dos maiores diferenciais do Iceberg: evoluir o schema é instantâneo — apenas os metadados são atualizados, sem tocar nos arquivos Parquet existentes.

In [9]:
spark.sql('ALTER TABLE local.superstore.pedidos ADD COLUMN customer_rating INT')

print('\u2705 Schema Evolution — coluna customer_rating adicionada sem reescrever dados')
spark.sql(
    'SELECT order_id, product_name, category, profit, customer_rating FROM local.superstore.pedidos'
).show(truncate=True)

✅ Schema Evolution — coluna customer_rating adicionada sem reescrever dados
+--------------+--------------------+---------------+-------+---------------+
|      order_id|        product_name|       category| profit|customer_rating|
+--------------+--------------------+---------------+-------+---------------+
|CA-2014-115812|Chromcraft Rectan...|      Furniture|85.3092|           NULL|
|CA-2014-115812|Eldon Expressions...|      Furniture|14.1694|           NULL|
|CA-2016-152156|Bush Somerset Col...|      Furniture|41.9136|           NULL|
|CA-2016-152156|Hon Deluxe Fabric...|      Furniture|219.582|           NULL|
|US-2024-NOVO01|Apple MacBook Pro...|     Technology| 499.99|           NULL|
|CA-2016-138688|Self-Adhesive Add...|Office Supplies| 6.8714|           NULL|
|CA-2014-167164|Fellowes Super St...|Office Supplies|   9.99|           NULL|
|CA-2014-143336|          Newell 341|Office Supplies| 2.4824|           NULL|
|US-2015-108966|Eldon Fold 'N Rol...|Office Supplies| 2.5164|     

## 10. Estrutura de arquivos no storage

In [10]:
print('Estrutura de arquivos Iceberg:')
pedidos_path = os.path.join(ICEBERG_PATH, 'superstore', 'pedidos')
for root, dirs, files in os.walk(pedidos_path):
    dirs.sort()
    level  = root.replace(pedidos_path, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in sorted(files):
        print(f'{indent}  {f}')

Estrutura de arquivos Iceberg:
pedidos/
  data/
    category=Furniture/
      .00000-10-9ee32f34-f249-408a-bdf6-2be0611e9cd2-0-00002.parquet.crc
      .00000-23-7f9a06a5-59d9-4d6a-b375-0af39ae6d642-0-00002.parquet.crc
      .00000-29-26ee4a96-b163-404d-9ffd-719667c56519-0-00002.parquet.crc
      .00000-63-0f29fe74-31b2-4060-b8ef-1facd142db28-0-00001.parquet.crc
      00000-10-9ee32f34-f249-408a-bdf6-2be0611e9cd2-0-00002.parquet
      00000-23-7f9a06a5-59d9-4d6a-b375-0af39ae6d642-0-00002.parquet
      00000-29-26ee4a96-b163-404d-9ffd-719667c56519-0-00002.parquet
      00000-63-0f29fe74-31b2-4060-b8ef-1facd142db28-0-00001.parquet
    category=Office+Supplies/
      .00000-10-9ee32f34-f249-408a-bdf6-2be0611e9cd2-0-00001.parquet.crc
      .00000-23-7f9a06a5-59d9-4d6a-b375-0af39ae6d642-0-00001.parquet.crc
      .00000-29-26ee4a96-b163-404d-9ffd-719667c56519-0-00001.parquet.crc
      00000-10-9ee32f34-f249-408a-bdf6-2be0611e9cd2-0-00001.parquet
      00000-23-7f9a06a5-59d9-4d6a-b375-0af39ae6